In [1]:
# %% [markdown]
# # 08_multi_model_comparison.ipynb
# Compares GPT-4o-mini vs GPT-4o on the v2 architecture, using the
# same question-level paired methodology as
# 06_statistical_reanalysis.py. Directly addresses Reviewer 2
# Concern #2: "The relative performance of the Semantic Layer and
# Text-to-SQL may differ for other models... this should be verified
# for the credit rating domain specifically."
#
# This does NOT re-verify exposure (0/15 columns is a property of
# build_v2_prompt() and the compiler, independent of which model is
# called -- already established exhaustively in
# 05a_exposure_final_check.py and unaffected by model choice). This
# notebook is purely about whether ACCURACY and FAILURE PATTERNS are
# model-dependent.
#
# Requires: results_phase3_v2_arch.csv (GPT-4o-mini, from
# 04_full_experiment.py) and results_phase3_v2_gpt4o.csv (GPT-4o,
# from 07_multi_model_experiment.py) both present.

# %%
print("SCRIPT VERSION: 2026-07-15-v1")

import pandas as pd
import numpy as np
from scipy import stats

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

df_mini = pd.read_csv('results_phase3_v2_arch.csv')
df_4o = pd.read_csv('results_phase3_v2_gpt4o.csv')

print(f"GPT-4o-mini: {len(df_mini)} records, {df_mini['question_id'].nunique()} questions, "
      f"{df_mini['iteration'].nunique()} iterations")
print(f"GPT-4o:      {len(df_4o)} records, {df_4o['question_id'].nunique()} questions, "
      f"{df_4o['iteration'].nunique()} iterations")

if df_mini['question_id'].nunique() != 60 or df_4o['question_id'].nunique() != 60:
    print("!! WARNING: one or both files do not cover all 60 questions.")
    print("   If a run is still in progress, results below reflect only")
    print("   the completed subset -- re-run this notebook after full")
    print("   completion for the final comparison.")

# %% [markdown]
# ## 1. Overall accuracy comparison

# %%
acc_mini = df_mini['is_correct'].mean()
acc_4o = df_4o['is_correct'].mean()

print("=" * 70)
print("OVERALL ACCURACY")
print("=" * 70)
print(f"GPT-4o-mini: {acc_mini:.1%}")
print(f"GPT-4o:      {acc_4o:.1%}")
print(f"Difference (GPT-4o - GPT-4o-mini): {acc_4o - acc_mini:+.1%}")

# %% [markdown]
# ## 2. Question-level paired comparison
#     (same methodology as 06_statistical_reanalysis.py's v1-vs-v2
#     comparison, applied here to mini-vs-4o)

# %%
def question_level_accuracy(df):
    return df.groupby('question_id')['is_correct'].mean()

q_mini = question_level_accuracy(df_mini)
q_4o = question_level_accuracy(df_4o)

common_qids = sorted(set(q_mini.index) & set(q_4o.index))
print(f"\nn = {len(common_qids)} question pairs in common")
if len(common_qids) != 60:
    print(f"!! Only {len(common_qids)} questions are common to both files "
          f"(expected 60) -- likely one run is incomplete.")

mini_aligned = q_mini.loc[common_qids].values
o4_aligned = q_4o.loc[common_qids].values
diff = o4_aligned - mini_aligned

print("=" * 70)
print("PAIRED COMPARISON: GPT-4o vs GPT-4o-mini (question-level)")
print("=" * 70)
print(f"GPT-4o-mini mean question-level accuracy: {mini_aligned.mean():.4f}")
print(f"GPT-4o mean question-level accuracy:      {o4_aligned.mean():.4f}")
print(f"Mean difference (4o - mini): {diff.mean():+.4f}")

t_stat, t_p = stats.ttest_rel(o4_aligned, mini_aligned)
try:
    w_stat, w_p = stats.wilcoxon(o4_aligned, mini_aligned)
except ValueError as e:
    w_stat, w_p = np.nan, np.nan
    print(f"(Wilcoxon skipped: {e})")

print(f"\nPaired t-test:        t={t_stat:.4f}, p={t_p:.4g}")
print(f"Wilcoxon signed-rank: W={w_stat:.4f}, p={w_p:.4g}")
print(f"\nQuestions where GPT-4o > GPT-4o-mini: {(diff > 0).sum()}, "
      f"mini > 4o: {(diff < 0).sum()}, tied: {(diff == 0).sum()}")

# %% [markdown]
# ## 3. Question-level bootstrap 95% CI

# %%
def question_level_bootstrap(values_a, values_b, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(values_a)
    boot_diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        boot_diffs[i] = values_b[idx].mean() - values_a[idx].mean()
    ci = np.percentile(boot_diffs, [2.5, 97.5])
    return ci

ci_diff = question_level_bootstrap(mini_aligned, o4_aligned)
print(f"95% CI for (GPT-4o - GPT-4o-mini) mean accuracy difference: "
      f"[{ci_diff[0]:+.4f}, {ci_diff[1]:+.4f}]")
print(f"CI contains zero: {ci_diff[0] <= 0 <= ci_diff[1]}")

# %% [markdown]
# ## 4. Failure taxonomy comparison
#     Key question for Reviewer 2: does a more capable model reduce
#     the compile_error rate specifically? If GPT-4o's compile_error
#     rate is substantially lower, this suggests the join-hint/
#     measure-usage failures documented in Section 5.4 are partly a
#     model-capability limitation rather than purely an architectural
#     one -- an important nuance for how the paper frames its
#     limitations.

# %%
def failure_taxonomy(df, label):
    total = len(df)
    n_correct = df['is_correct'].sum()
    n_compile = df['compile_error'].notna().sum()
    n_exec_all = df['execution_error'].notna().sum()
    n_timeout = df['execution_error'].str.contains('timeout', case=False, na=False).sum()
    n_exec_real = n_exec_all - n_timeout
    n_wrong = total - n_correct - n_compile - n_exec_all
    return pd.Series({
        'n': total,
        'correct_pct': n_correct / total,
        'compile_err_pct': n_compile / total,
        'exec_err_pct': n_exec_real / total,
        'timeout_pct': n_timeout / total,
        'wrong_result_pct': n_wrong / total,
    }, name=label)

tax_mini = failure_taxonomy(df_mini, 'GPT-4o-mini')
tax_4o = failure_taxonomy(df_4o, 'GPT-4o')

print("=" * 70)
print("FAILURE TAXONOMY COMPARISON")
print("=" * 70)
print(pd.concat([tax_mini, tax_4o], axis=1))

compile_err_reduction = tax_mini['compile_err_pct'] - tax_4o['compile_err_pct']
print(f"\nCompile error rate change (mini -> 4o): {-compile_err_reduction:+.1%}")
if compile_err_reduction > 0.05:
    print("GPT-4o shows a MEANINGFULLY LOWER compile error rate than")
    print("GPT-4o-mini (>5pp reduction) -- suggests the join-hint/measure-")
    print("usage failure patterns in Section 5.4 are at least partly a")
    print("model-capability limitation, not purely architectural.")
elif compile_err_reduction < -0.05:
    print("GPT-4o shows a HIGHER compile error rate than GPT-4o-mini --")
    print("unexpected; worth double-checking prompt compatibility across")
    print("models before drawing conclusions.")
else:
    print("Compile error rates are similar across models (<5pp difference)")
    print("-- suggests the failure patterns in Section 5.4 are largely")
    print("architectural (prompt/compiler design) rather than a specific")
    print("model's capability limitation, which strengthens the case that")
    print("fixing them requires semantic layer design changes rather than")
    print("simply using a stronger model.")

# %% [markdown]
# ## 5. Compile-error subtype comparison across models

# %%
def classify_compile_error(err):
    if pd.isna(err):
        return None
    if 'WHERE clause' in err:
        return 'measure-in-WHERE'
    elif 'MISMATCHED' in err:
        return 'join hint mismatched'
    elif 'does not correspond to any defined relationship' in err:
        return 'join hint unresolved'
    elif 'Excluded concept' in err:
        return 'excluded concept used'
    else:
        return 'other'

df_mini['compile_error_type'] = df_mini['compile_error'].apply(classify_compile_error)
df_4o['compile_error_type'] = df_4o['compile_error'].apply(classify_compile_error)

print("=" * 70)
print("COMPILE ERROR SUBTYPES: GPT-4o-mini vs GPT-4o")
print("=" * 70)
print("\nGPT-4o-mini:")
print(df_mini['compile_error_type'].value_counts())
print("\nGPT-4o:")
print(df_4o['compile_error_type'].value_counts())

# %% [markdown]
# ## 6. Category-level comparison

# %%
print("\n" + "=" * 70)
print("CATEGORY-LEVEL ACCURACY: GPT-4o-mini vs GPT-4o")
print("=" * 70)

cat_mini = df_mini.groupby('category')['is_correct'].mean()
cat_4o = df_4o.groupby('category')['is_correct'].mean()
cat_compare = pd.DataFrame({'GPT-4o-mini': cat_mini, 'GPT-4o': cat_4o})
cat_compare['diff'] = cat_compare['GPT-4o'] - cat_compare['GPT-4o-mini']
print(cat_compare)

# %% [markdown]
# ## 7. Difficulty-level comparison (does a stronger model raise the
#     challenging-tier floor?)

# %%
print("\n" + "=" * 70)
print("DIFFICULTY-LEVEL ACCURACY: GPT-4o-mini vs GPT-4o")
print("=" * 70)

diff_mini = df_mini.groupby('difficulty')['is_correct'].mean().reindex(['simple','moderate','challenging'])
diff_4o = df_4o.groupby('difficulty')['is_correct'].mean().reindex(['simple','moderate','challenging'])
diff_compare = pd.DataFrame({'GPT-4o-mini': diff_mini, 'GPT-4o': diff_4o})
diff_compare['diff'] = diff_compare['GPT-4o'] - diff_compare['GPT-4o-mini']
print(diff_compare)

print("\nChallenging-tier accuracy is the key indicator of whether the")
print("near-zero floor observed with GPT-4o-mini (Section 5.3's Jagged")
print("Frontier discussion) persists with a more capable model, or")
print("whether it is specific to GPT-4o-mini's capability level.")

# %% [markdown]
# ## 8. Latency comparison

# %%
print("\n" + "=" * 70)
print("LATENCY COMPARISON: GPT-4o-mini vs GPT-4o")
print("=" * 70)
lat_mini = df_mini['latency_ms'].agg(['mean', 'median', 'std'])
lat_4o = df_4o['latency_ms'].agg(['mean', 'median', 'std'])
print(pd.DataFrame({'GPT-4o-mini': lat_mini, 'GPT-4o': lat_4o}))

# %% [markdown]
# ## 9. Final summary for the manuscript

# %%
print("\n" + "=" * 70)
print("FINAL SUMMARY: GPT-4o-mini vs GPT-4o (v2 architecture)")
print("=" * 70)
print(f"""
  Metric                          GPT-4o-mini    GPT-4o
  ------------------------------  -----------    -----------
  Overall accuracy                 {acc_mini:.1%}          {acc_4o:.1%}
  Compile error rate               {tax_mini['compile_err_pct']:.1%}          {tax_4o['compile_err_pct']:.1%}
  Execution error rate             {tax_mini['exec_err_pct']:.1%}          {tax_4o['exec_err_pct']:.1%}
  Question-level paired test       --             t={t_stat:.2f}, p={t_p:.4g}
  Challenging-tier accuracy        {diff_mini['challenging']:.1%}           {diff_4o['challenging']:.1%}

  This addresses Reviewer 2 Concern #2 (single-LLM limitation): the
  v2 architecture's accuracy and failure taxonomy are now reported
  for two models in the same family (GPT-4o-mini, GPT-4o), showing
  whether the security-accuracy tradeoff documented in Section 5 is
  specific to GPT-4o-mini or generalizes to a more capable sibling
  model. Exposure (0/15 columns) is architecturally guaranteed by
  build_v2_prompt() and the local compiler regardless of which model
  is called, and is therefore not re-verified per-model here.
""")

print("Done. Use these tables directly for a new manuscript subsection")
print("addressing Reviewer 2 Concern #2 (e.g. Section 5.5 or an appendix).")

SCRIPT VERSION: 2026-07-15-v1
GPT-4o-mini: 300 records, 60 questions, 5 iterations
GPT-4o:      300 records, 60 questions, 5 iterations
OVERALL ACCURACY
GPT-4o-mini: 38.7%
GPT-4o:      42.0%
Difference (GPT-4o - GPT-4o-mini): +3.3%

n = 60 question pairs in common
PAIRED COMPARISON: GPT-4o vs GPT-4o-mini (question-level)
GPT-4o-mini mean question-level accuracy: 0.3867
GPT-4o mean question-level accuracy:      0.4200
Mean difference (4o - mini): +0.0333

Paired t-test:        t=0.8254, p=0.4125
Wilcoxon signed-rank: W=31.0000, p=0.5266

Questions where GPT-4o > GPT-4o-mini: 5, mini > 4o: 7, tied: 48
95% CI for (GPT-4o - GPT-4o-mini) mean accuracy difference: [-0.0400, +0.1167]
CI contains zero: True
FAILURE TAXONOMY COMPARISON
                  GPT-4o-mini   GPT-4o
n                    300.0000 300.0000
correct_pct            0.3867   0.4200
compile_err_pct        0.2100   0.2000
exec_err_pct           0.2767   0.1900
timeout_pct            0.0100   0.0100
wrong_result_pct       0.1167